In [89]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

data_path = "dataset/"
files = {}
for f in os.listdir(data_path):
    if f.endswith(".csv"):
        name = f.replace(".csv", "")
        files[name] = pd.read_csv(data_path + f)

In [90]:
sales = files['sales'].copy()
sales['Date'] = pd.to_datetime(sales['Date'])
sales = sales.sort_values('Date').reset_index(drop=True)
sales['GrossMargin'] = (sales['Revenue'] - sales['COGS']) / sales['Revenue']

TRAIN_END = '2022-12-31'
TEST_START = '2023-01-01'
TEST_END = '2024-07-01'

full_dates = pd.date_range(sales['Date'].min(), TEST_END, freq='D')
df = pd.DataFrame({'Date': full_dates})
df = df.merge(sales[['Date','Revenue','COGS','GrossMargin']], on='Date', how='left')

df['day_of_week'] = df['Date'].dt.dayofweek
df['day_of_month'] = df['Date'].dt.day
df['day_of_year'] = df['Date'].dt.dayofyear
df['week_of_year'] = df['Date'].dt.isocalendar().week.astype(int)
df['month'] = df['Date'].dt.month
df['year'] = df['Date'].dt.year

df['is_odd_year'] = (df['year'] % 2 != 0).astype(int)
df['is_august_odd'] = ((df['month'] == 8) & (df['is_odd_year'] == 1)).astype(int)
df['is_88'] = ((df['month'] == 8) & (df['day_of_month'] == 8)).astype(int)
df['is_99'] = ((df['month'] == 9) & (df['day_of_month'] == 9)).astype(int)
df['is_1010'] = ((df['month'] == 10) & (df['day_of_month'] == 10)).astype(int)
df['is_1111'] = ((df['month'] == 11) & (df['day_of_month'] == 11)).astype(int)
df['is_1212'] = ((df['month'] == 12) & (df['day_of_month'] == 12)).astype(int)

In [91]:
tet_dates = pd.to_datetime([
    '2012-01-23', '2013-02-10', '2014-01-31', '2015-02-19',
    '2016-02-08', '2017-01-28', '2018-02-16', '2019-02-05',
    '2020-01-25', '2021-02-12', '2022-02-01', '2023-01-22',
    '2024-02-10', '2025-01-29'
])

def get_days_until_tet(current_date):
    future_tets = tet_dates[tet_dates >= current_date]
    if len(future_tets) > 0:
        return (future_tets.min() - current_date).days
    return 365 

df['days_until_tet'] = df['Date'].apply(get_days_until_tet)

tet_window = []
for td in tet_dates:
    tet_window.extend(pd.date_range(td - pd.Timedelta(days=3), td + pd.Timedelta(days=7)))
df['is_tet_season'] = df['Date'].isin(tet_window).astype(int)

df['rev_lag_365'] = df['Revenue'].shift(365).fillna(df['Revenue'].shift(730))
df['gm_lag_365'] = df['GrossMargin'].shift(365).fillna(df['GrossMargin'].shift(730))
df['gm_lag_730'] = df['GrossMargin'].shift(730)
df['gm_was_negative_ly'] = (df['gm_lag_365'] < 0).astype(int)

In [92]:
train_df = df[(df['Date'] <= TRAIN_END) & (df['Revenue'].notna()) & (df['rev_lag_365'].notna())].copy()
test_df = df[df['Date'] >= TEST_START].copy()

train_df['rev_ratio_target'] = train_df['Revenue'] / (train_df['rev_lag_365'] + 1e-5)

# 1. Broad Calendar for Revenue (Smooth curves)
BASE_CALENDAR = [
    'day_of_week', 'day_of_month', 'day_of_year', 'week_of_year', 
    'month', 'year', 'days_until_tet'
]

# 2. Point-in-Time Triggers (Sharp needles)
PROMO_FLAGS = [
    'is_tet_season', 'is_88', 'is_99', 'is_1010', 'is_1111', 'is_1212'
]

# 3. Decoupled Feature Sets
# Revenue gets everything
feature_cols_rev = BASE_CALENDAR + PROMO_FLAGS + ['rev_lag_365']

# FIX: Gross Margin becomes a pure Boolean State Machine. No continuous lags. 
feature_cols_gm = PROMO_FLAGS + ['gm_was_negative_ly']

X_train_rev = train_df[feature_cols_rev]
y_train_rev = train_df['rev_ratio_target']
X_test_rev = test_df[feature_cols_rev]

X_train_gm = train_df[feature_cols_gm]
y_train_gm = train_df['GrossMargin']
X_test_gm = test_df[feature_cols_gm]

In [93]:
fig_stat = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=(
        'Raw Revenue Baseline', 
        'Stationary YoY Multiplier Target'
    )
)

fig_stat.add_trace(
    go.Scatter(x=train_df['Date'], y=train_df['Revenue'], mode='lines', 
               line=dict(color='steelblue', width=1)), 
    row=1, col=1
)
fig_stat.add_trace(
    go.Scatter(x=train_df['Date'], y=train_df['rev_ratio_target'], mode='lines', 
               line=dict(color='darkorange', width=1)), 
    row=2, col=1
)
fig_stat.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.8, row=2, col=1)

fig_stat.update_layout(height=600, width=1400, plot_bgcolor='white', hovermode='x unified', showlegend=False)
fig_stat.show()

rev_importance = pd.DataFrame({'Feature': feature_cols_rev, 'Importance': lgb_rev.feature_importances_}).sort_values(by='Importance')
gm_importance = pd.DataFrame({'Feature': feature_cols_gm, 'Importance': lgb_gm.feature_importances_}).sort_values(by='Importance')

fig_imp = make_subplots(rows=1, cols=2, horizontal_spacing=0.15, subplot_titles=('Revenue Features', 'Gross Margin Features'))
fig_imp.add_trace(go.Bar(y=rev_importance['Feature'], x=rev_importance['Importance'], orientation='h', marker_color='steelblue'), row=1, col=1)
fig_imp.add_trace(go.Bar(y=gm_importance['Feature'], x=gm_importance['Importance'], orientation='h', marker_color='purple'), row=1, col=2)

fig_imp.update_layout(height=500, width=1400, plot_bgcolor='white', showlegend=False)
fig_imp.show()

In [94]:
rev_ratio_preds = lgb_rev.predict(X_test_rev)
rev_preds = np.clip(rev_ratio_preds * X_test_rev['rev_lag_365'], 0, None)

gm_preds = np.clip(lgb_gm.predict(X_test_gm), -1.0, 1.0)
cogs_preds = np.clip(rev_preds * (1 - gm_preds), 0, None)

submission = pd.DataFrame({
    'Date': test_df['Date'].dt.strftime('%Y-%m-%d'),
    'Revenue': np.round(rev_preds, 2),
    'COGS': np.round(cogs_preds, 2),
})

submission.to_csv('submission_final.csv', index=False)

In [95]:
submission['Date'] = pd.to_datetime(submission['Date'])
submission['GrossMargin'] = (submission['Revenue'] - submission['COGS']) / submission['Revenue']

fig_final = make_subplots(
    rows=3, cols=1, 
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=('Revenue: Historical vs Predicted', 'COGS: Historical vs Predicted', 'Gross Margin: Historical vs Predicted')
)

fig_final.add_trace(go.Scatter(x=sales['Date'], y=sales['Revenue'], mode='lines', line=dict(color='steelblue', width=1.5)), row=1, col=1)
fig_final.add_trace(go.Scatter(x=submission['Date'], y=submission['Revenue'], mode='lines', line=dict(color='darkorange', width=2)), row=1, col=1)

fig_final.add_trace(go.Scatter(x=sales['Date'], y=sales['COGS'], mode='lines', line=dict(color='forestgreen', width=1.5)), row=2, col=1)
fig_final.add_trace(go.Scatter(x=submission['Date'], y=submission['COGS'], mode='lines', line=dict(color='firebrick', width=2)), row=2, col=1)

fig_final.add_trace(go.Scatter(x=sales['Date'], y=sales['GrossMargin'], mode='lines', line=dict(color='purple', width=1.5)), row=3, col=1)
fig_final.add_trace(go.Scatter(x=submission['Date'], y=submission['GrossMargin'], mode='lines', line=dict(color='magenta', width=2)), row=3, col=1)

for i in range(1, 4):
    fig_final.add_vline(x='2023-01-01', line_dash="dash", line_color="red", opacity=0.8, row=i, col=1)

fig_final.add_hline(y=0, line_dash="solid", line_color="black", opacity=0.8, row=3, col=1)

fig_final.update_layout(height=1000, width=1400, hovermode='x unified', plot_bgcolor='white', showlegend=False)
fig_final.show()
fig_final.write_html('final_predictions.html')